<a href="https://colab.research.google.com/github/afirdousi/pytorch-basics/blob/main/007_computer_vision_cnn_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms # (Read: https://pytorch.org/vision/stable/transforms.html)
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using PyTorch version = { torch.__version__ }")
print(f"Using device = { device }")  # We will be doing device agnostic code in this tutorial

Using PyTorch version = 2.0.1+cu118
Using device = cuda


In [3]:
# Intro to Convolutional Neural Networks  (aka as CNN)
# CNNs are also known as ConvNets.
# CNNs are known for their capabilities to find patterns in visual data

# General Structure of a CNN (pretty much same as normal vision model)

# 1 >Input Image
# 2 > Input Layer
# 3 > Pre Processing
# 4 > Convolution Layer (nn.ConvXd()) - Extracts/learns most important features from target images
# 5 > Hidden Activation / Non Linear Activation (nn.ReLU()) - Adds non-linearity to learned features (non-straight lines)
# 6 > Pooling Layer (nn.MaxPool2d()) - Reduces dimentionality of learned image features
# 7 > Output layer/linear layer (nn.Linear) - Takes learned features and outputs them in shape of target labels
# 8 > Output activation (torch.sigmoid) - Converts output logits to prediction probablities

# 4,5,6 can be combined in many different ways and can be repeated many times as a set of conv > relu > pooling

# Deeper CNN (Current research area)
# Having multiple layers of conv > relu > pooling
# Current: The accepted idea is the more layers you add to your CNN, the more deeper pattern it will learn

In [4]:
# Setup Training Data
train_data = datasets.FashionMNIST(
    root = "data", # which folder to download data to
    train = True, # all/most of built in datasets in PyTroch are already divided into train and testing datasets
    download = True, # do want to download?
    transform = torchvision.transforms.ToTensor(), # how do we want to transform the data
    target_transform = None #. how do we want to transform the labels/targets?
)

# Setup Test Data
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = torchvision.transforms.ToTensor(),
    target_transform= None
)

100%|██████████| 26421880/26421880 [00:02<00:00, 11966391.40it/s]


Extracting data/FashionMNIST/raw/train-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 203816.84it/s]


Extracting data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to data/FashionMNIST/raw



100%|██████████| 4422102/4422102 [00:01<00:00, 3739574.00it/s]


Extracting data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 23093344.38it/s]

Extracting data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to data/FashionMNIST/raw



In [5]:
# Learn what happens inside CNN on CNN Explainer: https://poloclub.github.io/cnn-explainer/
# Check: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
# Learn about hyperparameters of CNN on https://poloclub.github.io/cnn-explainer/ (search "Understanding Hyperparameters")

# Let's code
class FashionMNISTModelV2(nn.Module):
  """
  This model architecture will replicate the TinyVGG
  model from CNN explainer website
  """
  # Learn about Orginal VGG: https://medium.com/analytics-vidhya/vggnet-architecture-explained-e5c7318aa5b6

  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    # In CNN, things are generally referred to as Convolutional Block, think of a convolutional block as a set of conv > relu > pooling
    # deeper CNNs will have multiple conv blocks
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d( in_channels = input_shape, # Conv2d because we are working on a 2D image
                  out_channels = hidden_units,
                   kernel_size = 3, # this could be 3x3 or 3
                   stride = 1,
                   padding = 1), # these are all hyperparametrs used for conv 2d
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)  # takes the max value form a window of n x n witin an image, here n = 2
        # In general, one particular theme in deep neural networks is the images
        # will continue go down smaller and smaller the deeper you go in the network

    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.Conv2d( in_channels = hidden_units,
                  out_channels = hidden_units,
                   kernel_size = 3,
                   stride = 1,
                   padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2)
    )

    # finally we will have a classifier layer (where we flatten and classify into target labels)

    # compare this layer to previous simpler FashionMNIST models we made
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units * 0, # there is a trick to calculate this number which we have marked as 0 (more on it later)
                  out_features= output_shape)
    )

    def forward(self, x):
      x = self.conv.conv_block_1(x)
      print(x.shape)
      x = self.conv.conv_block_2(x)
      print(x.shape)
      x = self.conv.classifier(x)
      print(x.shape)

In [6]:
torch.manual_seed(42)

model_1 = FashionMNISTModelV2(input_shape= 1, # since we have black and white images (Vs CNN explainer guys have color videos)
                              hidden_units = 10,
                              output_shape = len(train_data.classes)).to(device)

/usr/local/lib/python3.10/dist-packages/torch/nn/init.py:405: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


In [7]:
## Stepping throuhg `nn.Conv2d()` (https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

# Lets create a dummy data (replicating data in cnn explainer website)
torch.manual_seed(42)
images = torch.randn(size =(32,3,64,64)) # quick note: in cnn explainer, the color channel "3" is shown as last param , 32 here is the batch size
test_image = images[0]

print(f"Image batch shape: {images.shape}")
print(f"Single batch shape: {test_image.shape}")
print(f"Test Image: {test_image}")

Image batch shape: torch.Size([32, 3, 64, 64])
Single batch shape: torch.Size([3, 64, 64])
Test Image: tensor([[[ 1.9269,  1.4873,  0.9007,  ...,  1.8446, -1.1845,  1.3835],
         [ 1.4451,  0.8564,  2.2181,  ...,  0.3399,  0.7200,  0.4114],
         [ 1.9312,  1.0119, -1.4364,  ..., -0.5558,  0.7043,  0.7099],
         ...,
         [-0.5610, -0.4830,  0.4770,  ..., -0.2713, -0.9537, -0.6737],
         [ 0.3076, -0.1277,  0.0366,  ..., -2.0060,  0.2824, -0.8111],
         [-1.5486,  0.0485, -0.7712,  ..., -0.1403,  0.9416, -0.0118]],

        [[-0.5197,  1.8524,  1.8365,  ...,  0.8935, -1.5114, -0.8515],
         [ 2.0818,  1.0677, -1.4277,  ...,  1.6612, -2.6223, -0.4319],
         [-0.1010, -0.4388, -1.9775,  ...,  0.2106,  0.2536, -0.7318],
         ...,
         [ 0.2779,  0.7342, -0.3736,  ..., -0.4601,  0.1815,  0.1850],
         [ 0.7205, -0.2833,  0.0937,  ..., -0.1002, -2.3609,  2.2465],
         [-1.3242, -0.1973,  0.2920,  ...,  0.5409,  0.6940,  1.8563]],

        [[-0.

In [9]:
# Let's give a random image to our model

# Create a single connv2d layer
conv_layer = nn.Conv2d(in_channels= 3, # since we have 3 color channels
                       out_channels= 10,
                       kernel_size=3, #  kernal_size=3 ==  kernal_size=(3x3) # kernal is also known as a filter
                       # kernal is a small convoluion, a small squar that moves left to right to learn patterns in image
                       # go to cnn explainer, click on an image and move/hover the small kernal across the image
                       # Got to CNN explainer and find the heading "Understanding Hyperparameters" and play with padding, kernal size and stride to learn visually
                       stride=1,
                       padding=0
                       )


# Pass the data through the convolution layer
conv_output = conv_layer(test_image)
conv_output

tensor([[[-2.8778e-01, -6.0596e-02, -5.6306e-02,  ...,  2.8654e-01,
           6.6224e-01, -2.3216e-01],
         [-9.8911e-01, -4.0099e-01,  4.1832e-01,  ...,  4.7459e-01,
          -1.8552e-01, -5.7622e-01],
         [-4.1340e-02, -2.3277e-01,  3.7418e-01,  ...,  2.8255e-02,
           1.4923e-01,  1.4236e-01],
         ...,
         [-8.0374e-01, -7.6687e-01, -5.9457e-02,  ...,  1.7452e-01,
           4.2594e-01, -4.8341e-01],
         [-1.4512e-01, -1.1566e-01,  6.1783e-01,  ...,  2.4126e-01,
          -3.6626e-01,  3.5645e-01],
         [ 3.6096e-02,  1.5214e-01,  2.3123e-01,  ...,  3.0904e-01,
          -4.9680e-01, -7.2258e-01]],

        [[-1.0853e+00, -1.6079e+00,  1.3346e-01,  ...,  2.1698e-01,
          -1.7643e+00,  2.5263e-01],
         [-8.2507e-01,  6.3866e-01,  1.8845e-01,  ..., -1.0936e-01,
           4.8068e-01,  8.4869e-01],
         [ 6.4927e-01, -4.2061e-03, -4.9991e-01,  ...,  5.8356e-01,
           2.4611e-01,  6.6233e-01],
         ...,
         [ 9.8860e-02,  1

In [10]:
conv_output.shape # [10, 62, 62] means our image has gone down from [ 3,64,64 ] to [ 10, 62, 62]

torch.Size([10, 62, 62])

In [11]:
# You can now play around with kernal and other hyperparameters to see how it changes the output layer
# For example:

conv_layer = nn.Conv2d(in_channels= 3,
                       out_channels= 10,
                       kernel_size=2,
                       stride=2,
                       padding=1
                       )


# Pass the data through the convolution layer
conv_output = conv_layer(test_image)
conv_output.shape

torch.Size([10, 33, 33])

In [12]:
# another example (play with more changing one or more variables at a time)
conv_layer = nn.Conv2d(in_channels= 3,
                       out_channels= 10,
                       kernel_size=3,
                       stride=3,
                       padding=1
                       )

conv_output = conv_layer(test_image)
conv_output.shape

torch.Size([10, 22, 22])